# Hotel Booking Cancellation & Overbooking Decision
## 02 — Data Quality and Leakage Analysis

### Objective
This notebook investigates missing values, duplicate records, suspicious values, categorical features, and potential target leakage before preprocessing.

**No raw data is modified in this notebook.**

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

## 2. Load the Raw Dataset

Update `DATA_PATH` if your local folder is different.

In [ ]:
DATA_PATH = r"C:\Users\ASUS\OneDrive\Desktop\AML Project\hotel_bookings.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
df.head()

## 3. Missing-Value Analysis

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values("Missing_Percentage", ascending=False)

missing_summary

Missing values are identified here only. Their treatment will be decided during preprocessing.

## 4. Duplicate-Record Analysis

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_percentage, 2), "%")

Duplicate-looking rows are not removed here because the dataset has no unique booking ID to prove that identical rows are duplicate transactions.

## 5. Numerical Range Inspection

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

numeric_range = pd.DataFrame({
    "Minimum": df[numeric_cols].min(),
    "Maximum": df[numeric_cols].max(),
    "Mean": df[numeric_cols].mean()
})

numeric_range

### Inspect Important Booking Variables

In [ ]:
columns_to_check = [
    "adults", "children", "babies",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adr", "lead_time"
]

df[columns_to_check].describe().T

## 6. Zero-Guest Bookings

In [ ]:
zero_guests = df[
    (df["adults"] == 0) &
    (df["children"].fillna(0) == 0) &
    (df["babies"] == 0)
]

print("Bookings with zero recorded guests:", len(zero_guests))
zero_guests.head()

Zero-guest records are flagged as suspicious but are not deleted at this stage.

## 7. Categorical Feature Inspection

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    print("\n" + "=" * 60)
    print("Column:", col)
    print("Unique values:", df[col].nunique(dropna=False))
    print(df[col].value_counts(dropna=False).head(10))

## 8. Target Leakage Analysis

Target leakage occurs when a model receives information that would not realistically be available at prediction time, especially information that reveals the final outcome.

Our goal is to predict cancellation **before the final reservation outcome is known**.

### Inspect Potential Leakage Features

In [ ]:
potential_leakage_cols = [
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list"
]

df[potential_leakage_cols].head(10)

## 9. `reservation_status` vs Target

In [ ]:
pd.crosstab(
    df["reservation_status"],
    df["is_canceled"],
    margins=True
)

### Leakage Finding — `reservation_status`

`reservation_status` contains final reservation states such as Canceled, Check-Out, or No-Show. This can reveal the outcome.

**Decision: exclude `reservation_status` from model features.**

## 10. `reservation_status_date`

In [ ]:
df[[
    "reservation_status_date",
    "reservation_status",
    "is_canceled"
]].head(15)

### Leakage Finding — `reservation_status_date`

This date is associated with reservation status and may only be known after the relevant outcome event.

**Decision: exclude `reservation_status_date` from the predictive feature set.**

## 11. Features Requiring Prediction-Time Consideration

These are not automatically labelled leakage:

- `assigned_room_type`
- `booking_changes`
- `days_in_waiting_list`

Whether they are valid depends on exactly when the prediction is made. They will be reviewed when defining the final feature set.

In [ ]:
review_cols = [
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list"
]

for col in review_cols:
    print("\nColumn:", col)
    print(df[col].value_counts(dropna=False).head(10))

## 12. Confirm Raw Data Has Not Been Modified

In [ ]:
print("Current dataset shape:", df.shape)
print("Target column present:", "is_canceled" in df.columns)
print("No cleaning or feature dropping has been performed.")

## Conclusions

### Data Quality
- Missing values require feature-specific treatment.
- Duplicate-looking rows require careful interpretation.
- Suspicious numerical records, including zero-guest bookings, need further investigation.
- Categorical variables will require suitable preprocessing.

### Clear Leakage Features
- `reservation_status`
- `reservation_status_date`

### Features Requiring Further Consideration
- `assigned_room_type`
- `booking_changes`
- `days_in_waiting_list`

### Next Step
The next notebook will define cleaning rules, construct the modelling dataset, and prepare the features for machine-learning preprocessing.